# Storage

Container-Dateisysteme sind nicht dauerhaft. PVCs fordern Speicher an, PVs repräsentieren bereitgestellten Speicher, StorageClasses beschreiben Provisionierung. CSI ist die standardisierte Schnittstelle zu Storage-Treibern; K3s local-path nutzt einen lokalen Provisioner, keinen CSI-Treiber.

![storage-architecture](storage-architecture.svg)

Statische Provisionierung: PV vorab anlegen. Dynamische Provisionierung: PV bei Bedarf erzeugen. ReadWriteOnce erlaubt schreibenden Mount auf einem Node; ReadWriteMany auf mehreren Nodes, sofern unterstützt.

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. StorageClass und PVC

WaitForFirstConsumer verzögert die Bindung bis zur Pod-Erstellung, damit die Node-Platzierung berücksichtigt werden kann.

In [ ]:
%%bash
set -eo pipefail
kubectl get storageclass
kubectl get storageclass local-path -o yaml
cat local-pvc-test.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f local-pvc-test.yml
kubectl get pvc pvc-test

## 2. Pod und persistente Daten

Der Pod bindet den PVC unter /data ein. Der Provisioner erzeugt das PV dynamisch.

In [ ]:
%%bash
set -eo pipefail
cat local-filesystem-app.yml
kubectl apply -f local-filesystem-app.yml
kubectl wait --for=condition=Ready pod/volpod --timeout=180s
kubectl get pvc pvc-test
kubectl get pv
kubectl get pod volpod -o wide

In [ ]:
%%bash
set -eo pipefail
kubectl exec volpod -- sh -c 'echo persistent > /data/demo.txt'
kubectl exec volpod -- cat /data/demo.txt

## 3. Pod ersetzen

Der PVC bleibt bestehen. Node-lokaler Speicher bindet den Ersatz-Pod an den passenden Node. Die Datei muss weiterhin vorhanden sein.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f local-filesystem-app.yml
kubectl apply -f local-filesystem-app.yml
kubectl wait --for=condition=Ready pod/volpod --timeout=180s
kubectl exec volpod -- cat /data/demo.txt
kubectl get pod volpod -o wide

## 4. Reclaim Policy und Aufräumen

Delete entfernt normalerweise das dynamische PV samt Speicher; Retain lässt es zur manuellen Verwaltung zurück. Erst Pod, dann PVC löschen.

In [ ]:
%%bash
set -eo pipefail
PV=$(kubectl get pvc pvc-test -o jsonpath='{.spec.volumeName}')
POLICY=$(kubectl get pv "$PV" -o jsonpath='{.spec.persistentVolumeReclaimPolicy}')
echo "$PV: $POLICY"
kubectl delete -f local-filesystem-app.yml
kubectl delete -f local-pvc-test.yml
if [ "$POLICY" = Delete ]; then kubectl wait --for=delete pv/"$PV" --timeout=120s; fi
kubectl get pv "$PV" --ignore-not-found

## 5. Ausblick StatefulSets

volumeClaimTemplates erzeugt pro Replikat einen eigenen PVC. Stabile Identität und Persistenz sind unterschiedliche Konzepte. local-path ist kein verteilter oder hochverfügbarer Speicher. Cloud-Beispiele bleiben Referenzmaterial.

## Fazit

PVC und PV entkoppeln Daten vom Lebenszyklus eines Pods. StorageClass und Reclaim Policy bestimmen Bereitstellung und Löschverhalten.